# M15 — LoRanPAC task-one numerical closure (train-only)

This audit closes the single failed numerical gate from M14. It reconstructs only task one for seed 4105 and a passing sentinel seed 4101, never creates `test.pt`, and never computes accuracy, logits, or predictions. QR is diagnostic and preserves the represented truncated system through the transformed core. M14 remains formally failed regardless of this result. Run every cell in order on a GPU runtime.

In [ ]:
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_COMMIT='13b0bd063c471ebddfd7e054d5f1a02a50eeeed4'
WORK_DIR='/content/SOHO-CL'
FEATURE_CACHE_DIR='/content/srq_m15_cifar_features'
OUTPUT_DIR='/content/srq_m15_output'
SOURCE_NAME='srq_generalization_m14_loranpac_multiseed_train_only.zip'
SOURCE_SHA='4beb726bf7f29f8569e5c6de630d90284abdf7ae53928471c27bba178d148b0c'
CONFIG='configs/srq_generalization_m15_loranpac_task1_closure.json'
RUNNER='tools/srq_generalization_m15.py'
CHECKPOINT_SHA='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CHECKPOINT_SIZE=346284714
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Exact checkout, dependencies, GPU, and canonical-LF source verification.
import hashlib,json,os,shutil,subprocess,sys
from pathlib import Path
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone',REPO_GIT_URL,WORK_DIR],check=True)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=WORK_DIR,check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Select Runtime -> Change runtime type -> GPU.'
def sha_raw(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def sha_source(path): return hashlib.sha256(Path(path).read_bytes().replace(b'\r\n',b'\n')).hexdigest()
EXPECTED={
 'configs/srq_generalization_m15_loranpac_task1_closure.json':'8cc0934faf1014658116fdb9c9977d081c42f20cbd109c22fcee167a0482b71c',
 'tools/srq_generalization_m15.py':'be2e521804e9059fd2b24d8e0aef871969448b39d216aa6639f105e5cca0b32d',
 'methods/frontends/loranpac.py':'b468d98981671876bbd223e62ab34ad8430d93f315662b0dabd36ab28ba980f7',
 'tools/experiment_runner.py':'b2c953eea312a98ce4146757fb46a7dd0e4ebe20aa139da59464921b11f8310c',
 'models/backbone.py':'941e449dc6e66ca4018fb0d3ab3218d97ec97f498b557ed220c8332e75850a46',
 'utils/data_utils.py':'3cf85993e231b068ad5ae2f96be608b2e50e9c52f98fb2387fd3badfb44b6764',
 'utils/train_utils.py':'e24983bd3042ad82ec069916ba2853cf1c818cb2911ce193710c8ccd70e86bda',
 'tests/test_srq_generalization_m15.py':'418e95947e912358a7e79d29f4084430c58d12fcbc59481343d0bf37b558a73e',
 'docs/research/SRQ_GENERALIZATION_M15_PROTOCOL.md':'d07de674efe99fe5f6794b48a82557b3b1bce7077986a336c043d3293c47744d'}
for path,expected in EXPECTED.items(): assert sha_source(path)==expected,(path,sha_source(path),expected)
assert subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()==REPO_COMMIT
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('GPU:',torch.cuda.get_device_name(0),'| M15 SOURCE LOCK: PASS')

In [ ]:
# Focused algebra, protocol, and LoRanPAC gates before data download.
command=[sys.executable,'-B','-m','pytest','-q','-p','no:cacheprovider','tests/test_srq_generalization_m15.py','tests/test_loranpac_analytic_frontend.py']
completed=subprocess.run(command)
assert completed.returncode==0,'M15 local gates failed; return the complete traceback.'
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('M15 LOCAL GATES: PASS')

In [ ]:
# Upload and verify the immutable M14 artifact.
from google.colab import files
os.chdir('/content')
uploaded=files.upload()
assert set(uploaded)=={SOURCE_NAME},f'Upload exactly {SOURCE_NAME}; got {list(uploaded)}'
SOURCE_ARTIFACT=str((Path('/content')/SOURCE_NAME).resolve())
assert sha_raw(SOURCE_ARTIFACT)==SOURCE_SHA,(sha_raw(SOURCE_ARTIFACT),SOURCE_SHA)
os.chdir(WORK_DIR)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('M14 ARTIFACT IDENTITY: PASS')

In [ ]:
# Download the locked checkpoint and processed CIFAR-100 source.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==CHECKPOINT_SIZE
assert sha_raw(CHECKPOINT_PATH)==CHECKPOINT_SHA
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
print('CHECKPOINT:',CHECKPOINT_PATH)
print('CIFAR ROOT:',CIFAR_ROOT)

In [ ]:
# Materialize TRAIN features only; held-out features are forbidden.
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size',str(CHECKPOINT_SIZE),'--backbone-checkpoint-sha256',CHECKPOINT_SHA,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_m15','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
metadata=json.loads((cache/'metadata.json').read_text())
assert metadata['feature_dim']==768 and metadata['finite'] is True
assert sha_raw(cache/'train.pt')=='ba53b82123964708123fe868a69d688bfdda4d66b1a3b7ea56329cc9ee9321dc'
print('TRAIN CACHE READY:',metadata.get('train_shape'),'| test.pt absent')

In [ ]:
# Four resumable seed/width SVD units produce eight budget records. Rerun this cell after an in-session interruption.
command=[sys.executable,'-u',RUNNER,'run','--config',CONFIG,'--source-m14-artifact',SOURCE_ARTIFACT,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir',OUTPUT_DIR,'--device','cuda','--require-clean-git']
print('M15 START/RESUME: 2 seeds x 2 widths; no prediction and no test data.',flush=True)
completed=subprocess.run(command)
RUN_RETURN_CODE=completed.returncode
unit_dir=Path(OUTPUT_DIR)/'units'
print('COMPLETED WIDTH UNITS:',sorted(path.name for path in unit_dir.glob('*.json')))
result_path=Path(OUTPUT_DIR)/'m15_results.json'
assert result_path.is_file(),'M15 stopped inside a width unit. If the runtime is alive, rerun this cell; completed width units are reused.'
result=json.loads(result_path.read_text())
print('STATUS:',result['status'])
print('SUMMARY:',json.dumps(result['summary'],indent=2))
print('GATES:',json.dumps(result['gates'],indent=2))

In [ ]:
# Numerical table and plot; no predictive metric is present.
import pandas as pd, matplotlib.pyplot as plt
frame=pd.read_csv(Path(OUTPUT_DIR)/'m15_numerical_metrics.csv')
display(frame)
fig,axes=plt.subplots(1,2,figsize=(11,4.2))
for (seed,budget),part in frame[frame['ridge']=='official_zero'].groupby(['seed','budget_target']):
    label=f'{seed}/{budget}'
    axes[0].plot(part['width'],part['raw_fp32_residual'],marker='o',label=label)
    axes[1].plot(part['width'],part['qr_fp64_core_backward_error'],marker='o',label=label)
axes[0].axhline(1e-3,color='black',linestyle='--',linewidth=1,label='M14 locked gate')
axes[0].set_ylabel('Raw projected-solve residual')
axes[1].set_ylabel('System-preserving FP64 core backward error')
for ax in axes:
    ax.set_xlabel('Width'); ax.set_yscale('log'); ax.grid(True,alpha=.25); ax.legend(fontsize=6)
fig.tight_layout()
plot_path=Path(OUTPUT_DIR)/'m15_task1_numerical_closure.svg'
fig.savefig(plot_path,format='svg'); plt.show(); plt.close(fig)
assert plot_path.is_file()

In [ ]:
# Export audit evidence. M14 remains FAIL even when M15 passes.
import zipfile
export=Path('/content/srq_generalization_m15_loranpac_task1_closure.zip')
members=[Path(OUTPUT_DIR)/'m15_results.json',Path(OUTPUT_DIR)/'m15_numerical_metrics.csv',Path(OUTPUT_DIR)/'m15_task1_numerical_closure.svg',Path(CONFIG),Path('docs/research/SRQ_GENERALIZATION_M15_PROTOCOL.md')]
members.extend(sorted((Path(OUTPUT_DIR)/'units').glob('*.json')))
manifest={}
with zipfile.ZipFile(export,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in members: archive.write(path,path.name); manifest[path.name]=sha_raw(path)
    archive.writestr('MANIFEST.json',json.dumps({'schema_version':1,'artifact':'M15 LoRanPAC task-one numerical closure','m14_status_remains':'FAIL_M14_LORANPAC_MULTISEED_TRAIN_ONLY','files':manifest},indent=2)+'\n')
print('ARTIFACT:',export,'SHA-256:',sha_raw(export),'bytes:',export.stat().st_size)
files.download(str(export))
assert RUN_RETURN_CODE==0 and result['status']=='PASS_M15_LORANPAC_TASK1_CLOSURE','Preserve the artifact and complete output; do not relax M14 or M15 gates.'